In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# Database credentials
username   = 'admin'
password   = 'admin%40123'
ip_address = '100.90.162.48'
port       = '5432'
db_name    = 'chicago_crime'

DB_URI = f'postgresql://{username}:{password}@{ip_address}:{port}/{db_name}'

new_engine = create_engine(DB_URI)

In [ ]:
from sqlalchemy.orm import sessionmaker

Session = sessionmaker(bind=new_engine)
session = Session()

In [ ]:
# Quick connection sanity-check
with new_engine.connect() as conn:
    result = conn.execute(text("SELECT count(*) FROM chicago_crime WHERE crime_type = 'THEFT'"))

print(result.fetchall())

## SQLCoder SQL Agent

Unlike the LangGraph agent in `test.ipynb` (which uses `llama3.1:8b` with function/tool calls),  
**SQLCoder is a specialised text-to-SQL completion model** — it does not support tool calls.  
Instead, the full database schema is injected directly into the prompt and the model emits plain SQL.

Pipeline:
```
get_schema  →  build_sqlcoder_prompt  →  SQLCoder LLM  →  extract_sql  →  db.run(sql)  →  (optional) chat LLM summary
```

All helper logic lives in `sqlcoder_agent.py`.

### Model

Pull the SQLCoder model with Ollama before running this notebook:
```bash
ollama pull sqlcoder          # ~4 GB — defog/sqlcoder-7b-2 quantised
# OR
ollama pull defog/sqlcoder-7b-2
```

> **Tip:** set `OLLAMA_SQLCODER_MODEL` below to whichever tag Ollama reports for `ollama list`.

In [1]:
from langchain_ollama import ChatOllama

OLLAMA_BASE_URL      = 'http://localhost:11434'   # local Ollama; change if remote
OLLAMA_SQLCODER_MODEL = 'mannix/defog-llama3-sqlcoder-8b'                # or 'defog/sqlcoder-7b-2'

# SQLCoder is a completion model; temperature=0 keeps output deterministic
sqlcoder_model = ChatOllama(
    model=OLLAMA_SQLCODER_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0,
)

print(f'SQLCoder model loaded: {OLLAMA_SQLCODER_MODEL}')

SQLCoder model loaded: mannix/defog-llama3-sqlcoder-8b


### (Optional) Chat Model for Natural-Language Summaries

After SQLCoder produces and executes SQL, a separate chat model (e.g. `llama3.1:8b`)  
can translate the raw rows into a readable answer.  
Skip this cell and pass `chat_model=None` to `run_sqlcoder_with_summary` if you don't need it.

In [2]:
OLLAMA_CHAT_MODEL = 'llama3.1:8b'    # any chat-capable Ollama model

chat_model = ChatOllama(
    model=OLLAMA_CHAT_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0,
)

print(f'Chat model loaded: {OLLAMA_CHAT_MODEL}')

Chat model loaded: llama3.1:8b


### Connecting to the SQL Database

In [3]:
from langchain_community.utilities import SQLDatabase

DB_USER     = 'admin'
DB_PASSWORD = 'admin%40123'
DB_HOST     = '100.90.162.48'
DB_PORT     = '5432'
DB_NAME     = 'chicago_crime'

LANGCHAIN_DB_URI = f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'

db = SQLDatabase.from_uri(LANGCHAIN_DB_URI)

print(f'Dialect          : {db.dialect}')
print(f'Available tables : {db.get_usable_table_names()}')

Dialect          : postgresql
Available tables : ['chicago_crime']


### Import SQLCoder Agent Helpers

In [4]:
import importlib
import sqlcoder_agent

importlib.reload(sqlcoder_agent)   # pick up any edits to sqlcoder_agent.py

from sqlcoder_agent import (
    get_schema,
    build_sqlcoder_prompt,
    extract_sql,
    run_sqlcoder,
    run_sqlcoder_with_summary,
)

print('Helper functions imported successfully.')

Helper functions imported successfully.


### Inspect the Schema Sent to SQLCoder

SQLCoder needs the full DDL in the prompt — inspect it here to make sure it looks correct.

In [5]:
schema = get_schema(db)
print(schema)


CREATE TABLE chicago_crime (
	"ID" BIGINT, 
	case_number TEXT, 
	"Date" TEXT, 
	"Block" TEXT, 
	"IUCR" TEXT, 
	crime_type TEXT, 
	"Description" TEXT, 
	location_description TEXT, 
	"Arrest" BOOLEAN, 
	"Domestic" BOOLEAN, 
	"Beat" BIGINT, 
	"District" BIGINT, 
	"Ward" DOUBLE PRECISION, 
	community_area DOUBLE PRECISION, 
	fbi_code TEXT, 
	x_coordinate DOUBLE PRECISION, 
	y_coordinate DOUBLE PRECISION, 
	"Year" BIGINT, 
	updated_on TEXT, 
	"Latitude" DOUBLE PRECISION, 
	"Longitude" DOUBLE PRECISION, 
	"Location" TEXT
)

/*
3 rows from chicago_crime table:
ID	case_number	Date	Block	IUCR	crime_type	Description	location_description	Arrest	Domestic	Beat	District	Ward	community_area	fbi_code	x_coordinate	y_coordinate	Year	updated_on	Latitude	Longitude	Location
14043280	JJ506901	11/30/2025 07:00:00 AM	025XX N NARRAGANSETT AVE	0810	THEFT	OVER $500	PARKING LOT / GARAGE (NON RESIDENTIAL)	False	False	2512	25	36.0	19.0	06	1133315.0	1916277.0	2025	12/08/2025 03:43:10 PM	41.926484632	-87.785553657	(

### Run the SQLCoder Agent

- `run_sqlcoder` — lean version: returns `sql` + raw `result` (no chat-model summary)  
- `run_sqlcoder_with_summary` — full version: adds a natural-language `answer` via `chat_model`

In [6]:
question = 'Which year had the most number of crimes?'

In [7]:
# SQLCoder only — no chat-model summary
out = run_sqlcoder(sqlcoder_model, db, question, verbose=True)

print(f'\n── Final SQL  : {out["sql"]}')
print(f'── DB result : {out["result"]}')
if out['error']:
    print(f'── Error     : {out["error"]}')


Question : Which year had the most number of crimes?
[Schema  : 1533 chars | Tables: ['chicago_crime']]

── SQLCoder raw output ─────────────────────────────
SELECT c."Year", COUNT(*) AS total_crimes FROM chicago_crime c GROUP BY c."Year" ORDER BY total_crimes DESC LIMIT 1;

── Extracted SQL ───────────────────────────────────
SELECT c."Year", COUNT(*) AS total_crimes FROM chicago_crime c GROUP BY c."Year" ORDER BY total_crimes DESC LIMIT 1

── Query Result ────────────────────────────────────
[(2023, 263278)]

── Final SQL  : SELECT c."Year", COUNT(*) AS total_crimes FROM chicago_crime c GROUP BY c."Year" ORDER BY total_crimes DESC LIMIT 1
── DB result : [(2023, 263278)]


In [12]:
question = 'What are the top 10 crime types by count?'

# Full pipeline with chat-model summary
out2 = run_sqlcoder_with_summary(
    sqlcoder_model=sqlcoder_model,
    db=db,
    question=question,
    chat_model=chat_model,   # set to None to skip summarisation
    verbose=True,
)

print(f'\n── Answer : {out2["answer"]}')


Question : What are the top 10 crime types by count?
[Schema  : 1533 chars | Tables: ['chicago_crime']]

── SQLCoder raw output ─────────────────────────────
SELECT c.crime_type, COUNT(*) AS count FROM chicago_crime c GROUP BY c.crime_type ORDER BY count DESC NULLS LAST LIMIT 10;

── Extracted SQL ───────────────────────────────────
SELECT c.crime_type, COUNT(*) AS count FROM chicago_crime c GROUP BY c.crime_type ORDER BY count DESC NULLS LAST LIMIT 10

── Query Result ────────────────────────────────────
[('THEFT', 307252), ('BATTERY', 253916), ('CRIMINAL DAMAGE', 161498), ('ASSAULT', 126595), ('MOTOR VEHICLE THEFT', 111268), ('DECEPTIVE PRACTICE', 100518), ('OTHER OFFENSE', 90113), ('ROBBERY', 49947), ('BURGLARY', 48559), ('WEAPONS VIOLATION', 47699)]

── Chat-model summary ──────────────────────────────
The top 10 crime types in Chicago by count are: THEFT (307,252), BATTERY (253,916), CRIMINAL DAMAGE (161,498), ASSAULT (126,595), MOTOR VEHICLE THEFT (111,268), DECEPTIVE PRACTICE (